# Benchmark: MitUNet y Raster2Seq contra el ground truth propio (PDV Nivel 1 y Nivel 2)

Objetivo: correr los 2 candidatos con mejor evidencia verificada del inventario de extracción geométrica (ver roadmap, 2026-08-05) contra el mismo ground truth `v13` ya usado para medir CubiCasa5K, Grounding DINO+SAM2 y floor-plan-walls (Roboflow).

**Runtime necesario: GPU** (Entorno de ejecución → Cambiar tipo de entorno → GPU, T4 o mejor).

**Aviso honesto**: ambos modelos requieren compilar módulos CUDA propios (deformable attention / diff_ras). Es un punto de falla típico en Colab por versión de CUDA/GCC — si una celda falla, copiar el error completo para diagnosticar, no asumir que el notebook está mal escrito de fondo.

**Qué subir cuando lo pida la Celda 2**: las 2 imágenes limpias del plano PDV (sin overlays propios) —
`archicheck_geometrico_pdv_05ago_0009_pag2-1.png` (Nivel 1) y `archicheck_geometrico_pdv_05ago_0009_pag2-2.png` (Nivel 2), ambas en `Fase 2/Desarrollos/Test/pdv/old/`.


## Celda 1 — Subir las 2 imágenes de prueba


In [ ]:
from google.colab import files
import os

os.makedirs('/content/test_images', exist_ok=True)
print('Selecciona las 2 imagenes (Nivel 1 y Nivel 2) cuando aparezca el boton:')
uploaded = files.upload()
for name, data in uploaded.items():
    with open(f'/content/test_images/{name}', 'wb') as f:
        f.write(data)
print('\nArchivos subidos:', list(uploaded.keys()))

# Asigna manualmente cual es Nivel 1 y cual Nivel 2 -- ajustar estos nombres
# a como se llamen los archivos que subiste si difieren.
NIVEL1_PATH = f'/content/test_images/{[n for n in uploaded if "pag2-1" in n][0]}'
NIVEL2_PATH = f'/content/test_images/{[n for n in uploaded if "pag2-2" in n][0]}'
print(f'Nivel 1: {NIVEL1_PATH}')
print(f'Nivel 2: {NIVEL2_PATH}')


## Celda 2 — Ground truth propio (v13), embebido acá para no depender de archivos externos

Coordenadas relativas (0-1) sobre la MISMA imagen subida arriba (mismo recorte usado para construir el ground truth, confirmado por ratio de aspecto contra `plano_nivelX_marcado_v12.png`).


In [ ]:
GT_NIVEL1 = {
    'puerta': [
        (0.2850,0.1351),(0.3675,0.1680),(0.4302,0.1577),(0.6526,0.1562),
        (0.1596,0.2731),(0.2113,0.3006),(0.3236,0.4085),(0.2911,0.4425),
        (0.4703,0.4831),(0.6622,0.4775),(0.8345,0.4820),(0.4699,0.6496),
        (0.5819,0.6497),(0.5433,0.7123),(0.7047,0.2259),(0.5185,0.8980),
        (0.1417,0.2989),
    ],
    'ventana': [
        (0.7047,0.1714),(0.7047,0.1979),(0.7047,0.2536),(0.7047,0.2802),
        (0.3883,0.2968),(0.5542,0.2940),(0.7500,0.3140),(0.3868,0.7310),
        (0.7178,0.7310),(0.2405,0.5439),(0.2405,0.6650),
    ],
    'muro': [
        (0.6745,0.0690),(0.8025,0.0695),(0.2194,0.0714),(0.9430,0.2004),
        (0.3298,0.3320),(0.6156,0.3980),(0.8771,0.4135),(0.2410,0.3166),
        (0.5757,0.4779),(0.7996,0.5415),(0.2433,0.7122),(0.6906,0.5737),
        (0.3682,0.5746),(0.2377,0.5927),(0.5864,0.6247),(0.3304,0.4661),
        (0.2451,0.4666),(0.5792,0.7129),(0.6978,0.4195),(0.8001,0.4407),
        (0.6208,0.4540),
    ],
}

GT_NIVEL2 = {
    'puerta': [
        (0.2440,0.3490),(0.2430,0.4870),(0.3119,0.4973),(0.2700,0.5240),
        (0.5120,0.4130),(0.6260,0.4470),(0.4603,0.5920),(0.6030,0.5950),
        (0.5960,0.6470),(0.3247,0.4612),
    ],
    'ventana': [
        (0.4350,0.3137),(0.2103,0.5258),(0.6750,0.4010),(0.7840,0.4470),
        (0.7779,0.5259),(0.3080,0.7313),(0.6800,0.7313),(0.6659,0.3171),
        (0.7548,0.3171),
    ],
    'muro': [
        (0.6211,0.3170),(0.7080,0.3170),(0.3155,0.3123),(0.6021,0.3611),
        (0.3036,0.4037),(0.8541,0.4164),(0.6863,0.4735),(0.2141,0.4743),
        (0.4016,0.4809),(0.6955,0.5734),(0.5331,0.5754),(0.7816,0.5899),
        (0.2150,0.6300),(0.5023,0.6406),(0.5048,0.6911),(0.4461,0.7307),
        (0.2384,0.3123),(0.2398,0.5098),(0.3078,0.3421),(0.5023,0.5966),
        (0.3618,0.5098),(0.4179,0.5261),(0.3450,0.5696),(0.4105,0.4296),
        (0.5698,0.4300),(0.2607,0.5696),(0.4185,0.5726),
    ],
}

print(f"GT Nivel 1: {len(GT_NIVEL1['puerta'])} puertas, {len(GT_NIVEL1['ventana'])} ventanas, {len(GT_NIVEL1['muro'])} puntos de muro")
print(f"GT Nivel 2: {len(GT_NIVEL2['puerta'])} puertas, {len(GT_NIVEL2['ventana'])} ventanas, {len(GT_NIVEL2['muro'])} puntos de muro")


---
# PARTE A — MitUNet (solo muros)


## Celda 3 — Clonar MitUNet e instalar dependencias


In [ ]:
%cd /content
!git clone https://github.com/aliasstudio/mitunet.git
%cd /content/mitunet
!pip install -q -r requirements.txt
!pip install -q segmentation-models-pytorch==0.5.0 albumentations


## Celda 4 — Descargar el checkpoint fine-tuneado

El README no da un link de descarga directo (indica bajarlo del propio repo). Si esta celda falla porque el archivo no está en el repo clonado, revisar el README actual del proyecto (`aliasstudio/mitunet`) para la ubicación real del checkpoint -- puede haberse movido a Releases o a un link externo desde que se investigó esto (2026-08-05).


In [ ]:
import os
CKPT_PATH = '/content/mitunet/experiments/models/mitunet_finetune_a6_mit_b4_tversky_8864_28E.pth'
if os.path.exists(CKPT_PATH):
    print('Checkpoint encontrado en el repo clonado:', CKPT_PATH)
else:
    print('NO se encontro el checkpoint en la ruta esperada.')
    print('Revisar el README de https://github.com/aliasstudio/mitunet para la ubicacion real')
    print('(Releases de GitHub, Google Drive, HuggingFace, etc.) y ajustar CKPT_PATH a mano.')
    !find /content/mitunet -iname '*.pth' -o -iname '*.pt'


## Celda 5 — Inferencia sobre Nivel 1 y Nivel 2, comparar contra el ground truth de muros


In [ ]:
import torch, cv2, numpy as np
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

# Arquitectura asumida a partir del nombre del checkpoint ('mit_b4') -- si la carga
# de pesos falla por mismatch de arquitectura, revisar en el repo que backbone/decoder
# usa realmente (probablemente Unet o UnetPlusPlus de smp con encoder mit_b4 = MixTransformer-B4).
model = smp.Unet(encoder_name='mit_b4', encoder_weights=None, in_channels=3, classes=1)
state = torch.load(CKPT_PATH, map_location=device)
if isinstance(state, dict) and 'state_dict' in state:
    state = state['state_dict']
model.load_state_dict(state, strict=False)
model.to(device).eval()

transform = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

def correr_mitunet(img_path, gt_muro_rel):
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    H, W = img_rgb.shape[:2]
    x = transform(image=img_rgb)['image'].unsqueeze(0).to(device)
    with torch.no_grad():
        pred = torch.sigmoid(model(x))[0, 0].cpu().numpy()
    mask = (pred > 0.5).astype(np.uint8)

    fig, axes = plt.subplots(1, 2, figsize=(16, 10))
    axes[0].imshow(img_rgb); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(mask, cmap='gray'); axes[1].set_title('Muros predichos (MitUNet)'); axes[1].axis('off')
    plt.tight_layout(); plt.show()

    aciertos = 0
    for (cx_rel, cy_rel) in gt_muro_rel:
        mx = int(cx_rel * 512)
        my = int(cy_rel * 512)
        # ventana de 5x5 px alrededor del punto GT, tolera error de pocos pixeles
        ventana = mask[max(0,my-2):my+3, max(0,mx-2):mx+3]
        if ventana.size and ventana.max() == 1:
            aciertos += 1
    recall = aciertos / len(gt_muro_rel) if gt_muro_rel else float('nan')
    print(f'  Puntos de muro GT dentro de la mascara predicha: {aciertos}/{len(gt_muro_rel)} (recall aprox: {recall:.1%})')
    return mask, recall

print('=== Nivel 1 ===')
mask_n1, recall_n1 = correr_mitunet(NIVEL1_PATH, GT_NIVEL1['muro'])
print('\n=== Nivel 2 ===')
mask_n2, recall_n2 = correr_mitunet(NIVEL2_PATH, GT_NIVEL2['muro'])


---
# PARTE B — Raster2Seq (muros + puertas + ventanas + recintos)


## Celda 6 — Clonar Raster2Seq, compilar módulos CUDA propios e instalar dependencias


In [ ]:
%cd /content
!git clone https://github.com/Cornell-VAILab/Raster2Seq.git
%cd /content/Raster2Seq
!pip install -q -r requirements.txt

%cd /content/Raster2Seq/models/ops
!sh make.sh

%cd /content/Raster2Seq/diff_ras
!python setup.py build develop

%cd /content/Raster2Seq
print('Si make.sh o setup.py fallaron: pegar el error completo, es el punto mas fragil de todo el notebook.')


## Celda 7 — Descargar el checkpoint CubiCasa5K (el más cercano en dominio a nuestro caso)


In [ ]:
%cd /content/Raster2Seq
!bash tools/download_checkpoints.sh cubicasa5k || echo 'Si este script falla, revisar tools/download_checkpoints.sh en el repo -- puede que la firma haya cambiado desde 2026-08-05.'
!find /content/Raster2Seq -iname '*.pth' -o -iname '*.pt'


## Celda 8 — Inferencia sobre Nivel 1 y Nivel 2

`predict.py` en este repo trabaja sobre una CARPETA de imágenes, no un archivo suelto -- se arma una carpeta `/content/r2s_input/` con las 2 imágenes de prueba. Los argumentos exactos (`--checkpoint`, `--image_size`, ruta del checkpoint descargado) están basados en el `argparse` del script leído el 2026-08-05 -- si Colab reporta un argumento desconocido, correr `!python predict.py --help` primero y ajustar.


In [ ]:
import os, shutil
os.makedirs('/content/r2s_input', exist_ok=True)
shutil.copy(NIVEL1_PATH, '/content/r2s_input/nivel1.png')
shutil.copy(NIVEL2_PATH, '/content/r2s_input/nivel2.png')

%cd /content/Raster2Seq
!python predict.py --help


In [ ]:
# Ajustar --checkpoint a la ruta real que imprimio la Celda 7 antes de correr esta celda.
# Ejemplo tentativo (verificar contra la salida real de --help de la celda anterior):
!python predict.py \
  --checkpoint checkpoints/cubicasa5k.pth \
  --output_dir /content/r2s_output \
  --save_pred --plot_pred

import os
print('Archivos generados:')
for root, dirs, fs_ in os.walk('/content/r2s_output'):
    for f in fs_:
        print(os.path.join(root, f))


## Celda 9 — Leer el JSON de salida, contar puertas/ventanas/recintos y comparar contra el ground truth

**Esta celda necesita ajustarse una vez que se vea la forma REAL del JSON de salida** (nombres de campos, IDs de categoría) -- el README no publicó el esquema exacto, solo confirmó que hay `category IDs per detected room` y `polygon coordinates`. Imprimir el JSON crudo primero, después mapear las categorías a puerta/ventana/muro/recinto antes de calcular recall.


In [ ]:
import json, glob

json_files = glob.glob('/content/r2s_output/**/*.json', recursive=True)
print('JSON encontrados:', json_files)

if json_files:
    with open(json_files[0]) as f:
        data = json.load(f)
    print(json.dumps(data, indent=2)[:3000])
    print('\n... (revisar la estructura completa arriba antes de calcular recall/precision)')
else:
    print('No se genero ningun JSON -- revisar la salida completa de la Celda 9 (cell-9) para el error real.')
